# TAXII Feed indicator enrichment
This notebook demonstrates useful ways of enriching threat indicators received from our TAXII feeds while using available filtering options.

### Used Spectra Intelligence classes
- **TAXIIFeed**
- **FileReputation**
- **NetworkReputation**
- **IPThreatIntelligence**
- **DomainThreatIntelligence**
- **URLThreatIntelligence**

### Credentials
Credentials are loaded from a local file instead of being written here in plain text.
To learn how to creat the credentials file, see the **Storing and using the credentials** section in the [README file](./README.md)

Let's set up the imports, credentials and the API clients that we will use.

In [33]:
import json
from datetime import datetime, timedelta
from ReversingLabs.SDK.ticloud import TAXIIFeed, FileReputation, NetworkReputation, IPThreatIntelligence, DomainThreatIntelligence, URLThreatIntelligence
from ReversingLabs.SDK.helper import NotFoundError, BadRequestError


CREDENTIALS = json.load(open("credentials.json"))
USERNAME = CREDENTIALS.get("ticloud").get("username")
PASSWORD = CREDENTIALS.get("ticloud").get("password")
USER_AGENT = json.load(open('../user_agent.json'))["user_agent"]

credentials = {
    "host": "https://data.reversinglabs.com",
    "username": USERNAME,
    "password": PASSWORD,
    "user_agent": USER_AGENT
    
}

taxii_client = TAXIIFeed(**credentials)
file_rep_client = FileReputation(**credentials)
net_rep_client = NetworkReputation(**credentials)
ip_ti = IPThreatIntelligence(**credentials)
domain_ti = DomainThreatIntelligence(**credentials)
url_ti = URLThreatIntelligence(**credentials)

# Define a time period of now - 1 day in the "zulu" string format
# You can try and change the number of days per need
two_days_ago = datetime.utcnow() - timedelta(days=2)
zulu_time = two_days_ago.strftime("%Y-%m-%dT%H:%M:%SZ")

#### Case 1: Enriching indicator objects with reputation data
In our first case we'll fetch file and network reputation data for received STIX indicators.  
The fetched reputation data will be appended to the indicator object under the `reputation` key.  
In this specific example we are fetching the following indicators:
- Trojan type
- Confidence of 90 and upwards
- Created during the last 24 hours

In [34]:
indicator_list = taxii_client.get_objects_aggregated(
    api_root="ransomware-relationships", 
    collection_id="5a715ba3-a450-457c-a086-dd9901ad1fad", 
    confidence=">=90",
    stix_types="indicator",
    labels=["Trojan"],
    added_after=zulu_time,
    max_results=100
)

for indicator in indicator_list:
    if "file:hashes" in indicator["pattern"]:
        try:
            reputation_resp = file_rep_client.get_file_reputation(indicator["name"])
            indicator["reputation"] = reputation_resp.json()
        except NotFoundError:
            continue
        
    else:
        try:
            reputation_resp = net_rep_client.get_network_reputation([indicator["name"]])
            indicator["reputation"] = reputation_resp.json()
        except NotFoundError:
            continue

print(json.dumps(indicator_list))

[{"type": "indicator", "spec_version": "2.1", "id": "indicator--90d6f8a7-9713-5b40-a6cd-ed027fb6d48d", "created": "2025-10-06T13:12:19Z", "modified": "2025-10-06T13:12:19Z", "valid_from": "2025-10-06T13:12:19Z", "valid_until": "2025-11-05T13:12:19Z", "created_by_ref": "identity--ef2a68d1-a8d2-5cb4-973d-1b9e0858c41e", "name": "2.180.121.180", "description": "Malware file was observed to contact an IP", "confidence": 100, "indicator_types": ["malicious-activity"], "pattern_type": "stix", "pattern": "[ipv4-addr:value = '2.180.121.180']", "labels": ["ReversingLabs", "TCP-40500", "AS58224::IR::TCI", "Phorpiex", "Trojan", "Early"], "external_references": [], "kill_chain_phases": [{"kill_chain_name": "rl-ransomware-kill-chain", "phase_name": "early"}], "revoked": false, "reputation": {"rl": {"entries": [{"type": "ip", "requested_network_location": "2.180.121.180", "associated_malware": false, "last_seen": "2025-10-08T09:46:53", "first_seen": "2025-10-08T09:46:52", "third_party_reputations": {

Here we can see the indicator list enriched with file and network reputation data.

#### Case 2: Enriching network indicators with network threat intelligence data
In our second case we'll remove file indicators and enrich network indicators with network threat intelligence data.  
The fetched network threat intelligence data will be appended to the indicator object under the `network_threat_intelligence` key.  
In this specific example we are fetching the following indicators:
- Ursnif family
- Confidence of 90 and upwards
- Created during the last 24 hours

In [35]:
indicator_list = taxii_client.get_objects_aggregated(
    api_root="ransomware-relationships", 
    collection_id="5a715ba3-a450-457c-a086-dd9901ad1fad", 
    confidence=">=90",
    stix_types="indicator",
    labels=["Trojan"],
    added_after=zulu_time,
    max_results=200
)

indicator_list = [indicator for indicator in indicator_list if "file:hashes" not in indicator["pattern"]]

for indicator in indicator_list:
    try:
        if "domain-name:value" in indicator["pattern"]:
            ti_resp = domain_ti.get_domain_report(indicator["name"])
    
        elif "url:value" in indicator["pattern"]:
            ti_resp = url_ti.get_url_report(indicator["name"])
    
        elif "ipv4-addr:value" in indicator["pattern"]:
            ti_resp = ip_ti.get_ip_report(indicator["name"])
    
        else:
            continue
        
        indicator["network_threat_intelligence"] = ti_resp.json()
    
    except (BadRequestError, NotFoundError):
        continue

print(json.dumps(indicator_list))

[{"type": "indicator", "spec_version": "2.1", "id": "indicator--90d6f8a7-9713-5b40-a6cd-ed027fb6d48d", "created": "2025-10-06T13:12:19Z", "modified": "2025-10-06T13:12:19Z", "valid_from": "2025-10-06T13:12:19Z", "valid_until": "2025-11-05T13:12:19Z", "created_by_ref": "identity--ef2a68d1-a8d2-5cb4-973d-1b9e0858c41e", "name": "2.180.121.180", "description": "Malware file was observed to contact an IP", "confidence": 100, "indicator_types": ["malicious-activity"], "pattern_type": "stix", "pattern": "[ipv4-addr:value = '2.180.121.180']", "labels": ["ReversingLabs", "TCP-40500", "AS58224::IR::TCI", "Phorpiex", "Trojan", "Early"], "external_references": [], "kill_chain_phases": [{"kill_chain_name": "rl-ransomware-kill-chain", "phase_name": "early"}], "revoked": false, "network_threat_intelligence": {"rl": {"requested_ip": "2.180.121.180", "third_party_reputations": {"statistics": {"malicious": 0, "clean": 0, "suspicious": 0, "undetected": 12, "total": 12}, "sources": [{"source": "levelblue"

#### Case 3: Enriching indicators with related indicator data
Now, let's do one request to get indicators, and one to get relationships.  
Then we will iterate through fetched relationships to see if they mention that any of our indicators are related to some other indicator. 
If it is, we will fetch the related indicator and append its data to our original indicator.

In [39]:
one_day_ago = datetime.utcnow() - timedelta(days=1)
zulu_time = one_day_ago.strftime("%Y-%m-%dT%H:%M:%SZ")

indicator_list = taxii_client.get_objects_aggregated(
    api_root="ransomware-relationships", 
    collection_id="5a715ba3-a450-457c-a086-dd9901ad1fad", 
    confidence=">=90",
    stix_types="indicator",
    labels=["Trojan"],
    added_after=zulu_time,
    max_results=200
)

relationship_list = taxii_client.get_objects_aggregated(
    api_root="ransomware-relationships", 
    collection_id="5a715ba3-a450-457c-a086-dd9901ad1fad", 
    stix_types="relationship",
    added_after=zulu_time,
    max_results=200
)

def create_related_indicator(indicator_id):
    indicator_resp = taxii_client.get_objects(
        api_root="ransomware-relationships", 
        collection_id="5a715ba3-a450-457c-a086-dd9901ad1fad", 
        match_id=indicator_id
    )
    
    resulting_indicator = indicator_resp.json().get("objects")[0]
    
    indicator_obj = {
        "id": resulting_indicator.get("id"),
        "name": resulting_indicator.get("name"),
        "confidence": resulting_indicator.get("confidence"),
        "pattern": resulting_indicator.get("pattern"),
        "labels": resulting_indicator.get("labels")
    }
    
    return indicator_obj

for indicator in indicator_list:
    indicator["related_indicators"] = []
    
    for relationship in relationship_list:
        if indicator["id"] == relationship["source_ref"]:
            related_indicator = create_related_indicator(relationship["target_ref"])
            indicator["related_indicators"].append(related_indicator) 
            
        elif indicator["id"] == relationship["target_ref"]:
            related_indicator = create_related_indicator(relationship["source_ref"])
            indicator["related_indicators"].append(related_indicator)

print(json.dumps(indicator_list))

[{"type": "indicator", "spec_version": "2.1", "id": "indicator--df39eab5-5e06-5710-a939-11ec48b57392", "created": "2025-10-07T13:12:40Z", "modified": "2025-10-07T13:12:40Z", "valid_from": "2025-10-07T13:12:40Z", "valid_until": "2025-11-06T13:12:40Z", "created_by_ref": "identity--ef2a68d1-a8d2-5cb4-973d-1b9e0858c41e", "name": "383bb9712a860c6582f15b97e0420a605f99fd96", "description": "Malware file activity was observed", "confidence": 100, "indicator_types": ["malicious-activity"], "pattern_type": "stix", "pattern": "[file:hashes.'SHA-1' = '383bb9712a860c6582f15b97e0420a605f99fd96' OR file:hashes.'MD5' = '82e5d008eeb658853fd07b22ef38c402' OR file:hashes.'SHA-256' = '1369e6e87985030f0ba01f1809a0a6247b759693c7b19e87fc15826fa841b65f']", "labels": ["ReversingLabs", "Quasar", "Trojan", "Early"], "external_references": [{"source_name": "mitre", "external_id": "T1003", "description": "OS Credential Dumping", "url": "https://attack.mitre.org/techniques/T1003/"}, {"source_name": "mitre", "extern